In [58]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/thoughtvector/customer-support-on-twitter/sample.csv
/kaggle/input/datasets/thoughtvector/customer-support-on-twitter/twcs/twcs.csv


In [13]:
print(os.listdir('/kaggle/input/datasets/'))


['thoughtvector']


In [59]:
DATA_PATH = '/kaggle/input/datasets/thoughtvector/customer-support-on-twitter/twcs/twcs.csv'

assert os.path.exists(DATA_PATH), f'File not found: {DATA_PATH}'

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nDtypes:\n{df.dtypes}')
df.head()

Shape: (2811774, 7)

Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

Dtypes:
tweet_id                     int64
author_id                   object
inbound                       bool
created_at                  object
text                        object
response_tweet_id           object
in_response_to_tweet_id    float64
dtype: object


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messages and no one is responding as usual,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [46]:
df['author_id'].value_counts()

author_id
AmazonHelp      169840
AppleSupport    106860
Uber_Support     56270
SpotifyCares     43265
Delta            42253
                 ...  
823870               1
823817               1
823815               1
823816               1
823814               1
Name: count, Length: 702777, dtype: int64

In [60]:
AMAZON_HELP_ID='AmazonHelp'

In [48]:
amazon = df[df["author_id"] == AMAZON_HELP_ID].copy()

# AmazonHelp responses
responses = amazon[amazon["inbound"] == False]

# Customer tweets that AmazonHelp responded to
customer = df[df["tweet_id"].isin(responses["in_response_to_tweet_id"])]

In [49]:
pairs = responses.merge(
    customer[["tweet_id", "text"]],
    left_on="in_response_to_tweet_id",
    right_on="tweet_id",
    suffixes=("_response", "_query")
)

pairs = pairs[["text_query", "text_response"]]
pairs.columns = ["user_query", "response"]

In [52]:
len(pairs)

168823

In [55]:
pairs[100:110]

,user_query,response
100,@AmazonHelp infórmese... no voy a perder más mi tiempo...,@116616 ¿Puedes explicarnos bien el inconveniente y mandarnos los detalles en este enlace por favor? https://t.co/kUkXHFkYWd ^JJ
101,@116618 why your videos on Germany breaks too?,@116617 Have you already tried a restart? Kind regards ^NW
102,"@AmazonHelp Many times, but still failing, and it happen with some show",@116617 Is this still ongoing? What devices are you using? ^AN
103,@AmazonHelp Sadly yes,@116617 What devices are you using for Amazon Video? ^AN
104,"@AmazonHelp Mac book pro, with Chrome",@116617 In this case please contact my colleagues via: https://t.co/LFLW8z8jr2 They will help you! Regards ^DW
105,"@AmazonHelp Mac book pro, with Chrome",@116617 Hi! I just wanted to ask if my colleagues could assist you? Regards ^TA
106,"@AmazonHelp I need to change the language of my account to be able to get support, which I don't have the time to do, there's any other way to contact?",@116617 Changing language is just a simple click: https://t.co/CXXk5VglR4 You can reach our support here: https://t.co/aAfOxTGpe1 ^UK
107,@115830 you pay for prime expecting next day delivery. Then receive 2 emails half hr apart 'failed delivery' but your sitting at home 😡,"@116619 I'm sorry your parcel wasn't delivered. Please reach us here: https://t.co/XRh1cqD2lT, so we can look into this with you. ^MO"
108,So sad @AmazonHelp @115830 ruined his FIRST Halloween by failing to deliver his tiny costume 2 days running #whypayprime? #amazon #prime,@116830 Oh no! Did we miss the expected delivery date from your confirmation e-mail here: https://t.co/OmqrPtDrzj Please let us know. ^KL
109,@115830 website impossible to navigate - all I want to do is email about a faulty item - 20 min going round in circles. Infuriating 😡😡😡,@116845 I'm sorry for the trouble! Let's go over your available options here: https://t.co/JzP7hlA23B\n^LL


In [63]:
import pandas as pd

# 1. Specify the exact date format instead of 'mixed'
# (Replace this example string with your actual date format)
EXACT_DATE_FORMAT = "%Y-%m-%d %H:%M:%S" 

# 2. Vectorized ID cleaning (avoiding redundant copies)
id_cols = ["tweet_id", "in_response_to_tweet_id", "response_tweet_id"]

for col in id_cols:
    # Convert floats/strings cleanly to nullable integers, then to string
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")
        .astype("Int64")
        .astype(str)
        .replace("<NA>", "")
    )

# 3. Exact date parsing (10x-50x faster)
df["created_at"] = pd.to_datetime(
    df["created_at"],
    format=EXACT_DATE_FORMAT,
    errors="coerce"
)

In [64]:
tweet_lookup = df.set_index("tweet_id")

In [65]:
tweet_lookup.loc["12345"]

author_id                                                                                                                                                                                                        118458
inbound                                                                                                                                                                                                            True
created_at                                                                                                                                                                                                          NaT
text                       @ChaseSupport Was looking to avail my coupon for 300$ for opening chase account But the bank was ill staffed and my account opening postponed to tomorrow. My coupon expires today. Plz help
response_tweet_id                                                                                                                       

In [66]:
amazon_tweets = df[
    df["author_id"] == "AmazonHelp"
].copy()

print("AmazonHelp tweets:", len(amazon_tweets))

AmazonHelp tweets: 169840


In [67]:
amazon_ids = set(
    amazon_tweets["tweet_id"]
)

In [70]:
parent_ids = set(
    amazon_tweets[
        amazon_tweets["in_response_to_tweet_id"] != ""
    ]["in_response_to_tweet_id"]
)

In [71]:
def parse_ids(value):
    if not value:
        return []
    
    return [
        x.strip()
        for x in value.split(",")
        if x.strip()
    ]


child_ids = set()

for value in amazon_tweets["response_tweet_id"]:
    child_ids.update(parse_ids(value))

In [72]:
related_ids = amazon_ids | parent_ids | child_ids

print("Amazon tweets:", len(amazon_ids))
print("Parent tweets:", len(parent_ids))
print("Child tweets:", len(child_ids))
print("Initial related tweets:", len(related_ids))

Amazon tweets: 169840
Parent tweets: 155445
Child tweets: 74131
Initial related tweets: 343902


In [73]:
all_ids = set(related_ids)

while True:

    current = df[
        df["tweet_id"].isin(all_ids)
    ]

    new_ids = set()

    # Parent relationships
    for value in current["in_response_to_tweet_id"]:
        if value:
            new_ids.add(value)

    # Child relationships
    for value in current["response_tweet_id"]:
        new_ids.update(parse_ids(value))

    new_ids -= all_ids

    if not new_ids:
        break

    all_ids.update(new_ids)

    print(
        "Added:",
        len(new_ids),
        "| Total:",
        len(all_ids)
    )

Added: 6862 | Total: 350764
Added: 1475 | Total: 352239
Added: 553 | Total: 352792
Added: 306 | Total: 353098
Added: 166 | Total: 353264
Added: 97 | Total: 353361
Added: 58 | Total: 353419
Added: 44 | Total: 353463
Added: 31 | Total: 353494
Added: 25 | Total: 353519
Added: 22 | Total: 353541
Added: 16 | Total: 353557
Added: 12 | Total: 353569
Added: 8 | Total: 353577
Added: 6 | Total: 353583
Added: 6 | Total: 353589
Added: 6 | Total: 353595
Added: 5 | Total: 353600
Added: 4 | Total: 353604
Added: 4 | Total: 353608
Added: 4 | Total: 353612
Added: 4 | Total: 353616
Added: 4 | Total: 353620
Added: 4 | Total: 353624
Added: 4 | Total: 353628
Added: 4 | Total: 353632
Added: 4 | Total: 353636
Added: 4 | Total: 353640
Added: 4 | Total: 353644
Added: 4 | Total: 353648
Added: 4 | Total: 353652
Added: 4 | Total: 353656
Added: 4 | Total: 353660
Added: 2 | Total: 353662
Added: 2 | Total: 353664
Added: 1 | Total: 353665
Added: 1 | Total: 353666
Added: 1 | Total: 353667
Added: 1 | Total: 353668
Added

In [74]:
amazon_related = df[
    df["tweet_id"].isin(all_ids)
].copy()

print(
    "Amazon-related tweets:",
    len(amazon_related)
)

Amazon-related tweets: 353436


In [77]:
parent_map = dict(
    zip(
        amazon_related["tweet_id"],
        amazon_related["in_response_to_tweet_id"]
    )
)

In [78]:
def find_root(tweet_id, parent_map):

    visited = set()
    current = tweet_id

    while True:

        if current in visited:
            return current

        visited.add(current)

        parent = parent_map.get(current, "")

        if not parent or parent not in parent_map:
            return current

        current = parent

In [79]:
amazon_related["root_id"] = (
    amazon_related["tweet_id"]
    .apply(lambda x: find_root(x, parent_map))
)

In [80]:
conversation_sizes = (
    amazon_related
    .groupby("root_id")
    .size()
    .sort_values(ascending=False)
)

conversation_sizes.describe()

count    82556.000000
mean         4.281167
std          4.615211
min          2.000000
25%          2.000000
50%          3.000000
75%          5.000000
max        448.000000
dtype: float64

In [81]:
conversation_sizes.head(20)

root_id
395394     448
5159       428
19137      200
964970     155
1208753    152
665469     136
35857      122
279455     121
851605     114
21428      114
12549      114
1041201    112
397305     112
861246     109
865815     109
258801      96
128087      95
128093      93
421372      91
179225      89
dtype: int64

In [82]:
def is_amazon_case(group):

    has_amazon = (
        group["author_id"] == "AmazonHelp"
    ).any()

    has_customer = (
        group["author_id"] != "AmazonHelp"
    ).any()

    return has_amazon and has_customer

In [84]:
valid_roots = []

for root_id, group in amazon_related.groupby("root_id"):

    if is_amazon_case(group):
        valid_roots.append(root_id)

print("Valid conversations:", len(valid_roots))

Valid conversations: 82556


In [85]:
conversations = amazon_related[
    amazon_related["root_id"].isin(valid_roots)
].copy()

In [86]:
conversations = conversations.sort_values(
    ["root_id", "created_at"]
)

In [87]:
def format_conversation(group):

    lines = []

    group = group.sort_values("created_at")

    for _, row in group.iterrows():

        speaker = (
            "AmazonHelp"
            if row["author_id"] == "AmazonHelp"
            else "Customer"
        )

        lines.append(
            f"{speaker}: {row['text']}"
        )

    return "\n".join(lines)

In [88]:
case_conversations = (
    conversations
    .groupby("root_id")
    .apply(format_conversation)
    .reset_index()
)

case_conversations.columns = [
    "conversation_id",
    "conversation"
]

/tmp/ipykernel_58/821385304.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(format_conversation)


In [89]:
case_conversations.head()

,conversation_id,conversation
0,1000119,"AmazonHelp: @356779 I'm sorry for the poor experience! When you contacted us, what options or insight were we able to offer? ^WT\nCustomer: @AmazonHelp It's all good now 😊 I chatted in &amp; it tu..."
1,1000123,AmazonHelp: @356780 Oh no! Please give us a call here: https://t.co/JzP7hlA23B so we can take a look at this with you! ^TR\nCustomer: @115830 I just inadvertently bought a Kindle book on my accoun...
2,1000125,"AmazonHelp: @356781 I understand your concern, Nehal. I'll pass along your feedback to the team concerned for review. ^HD\nCustomer: @119625 Please upload south movies in hindi audio"
3,1000127,AmazonHelp: @356782 I'm sorry for the subtitle issues. Please reach out to us here so we can look into this: https://t.co/hApLpMlfHN ^BH\nCustomer: @43895 watching Loveless on Amazon prime cuz of ...
4,1000130,"AmazonHelp: @356783 Oh no! That's not supposed to happen. Is there a particular error showing, Tim? Let us know, we are here to help! ^SA\nCustomer: @AmazonHelp For a while, every time I tried to ..."


In [90]:
def get_customer_messages(group):

    group = group.sort_values("created_at")

    messages = group[
        group["author_id"] != "AmazonHelp"
    ]["text"].dropna()

    return "\n".join(messages)

In [91]:
customer_problems = (
    conversations
    .groupby("root_id")
    .apply(get_customer_messages)
    .reset_index()
)

customer_problems.columns = [
    "conversation_id",
    "customer_problem"
]

/tmp/ipykernel_58/1401570537.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_customer_messages)


In [92]:
def get_amazon_responses(group):

    group = group.sort_values("created_at")

    messages = group[
        group["author_id"] == "AmazonHelp"
    ]["text"].dropna()

    return "\n".join(messages)

In [94]:
amazon_responses = (
    conversations
    .groupby("root_id")
    .apply(get_amazon_responses)
    .reset_index()
)

amazon_responses.columns = [
    "conversation_id",
    "amazon_responses"
]

/tmp/ipykernel_58/166085075.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_amazon_responses)


In [95]:
cases = (
    case_conversations
    .merge(
        customer_problems,
        on="conversation_id"
    )
    .merge(
        amazon_responses,
        on="conversation_id"
    )
)

In [96]:
conversation_stats = (
    conversations
    .groupby("root_id")
    .agg(
        turn_count=("tweet_id", "count"),
        customer_turns=(
            "inbound",
            lambda x: x.sum()
        ),
        first_timestamp=("created_at", "min"),
        last_timestamp=("created_at", "max")
    )
    .reset_index()
)

conversation_stats = conversation_stats.rename(
    columns={"root_id": "conversation_id"}
)

In [97]:
cases = cases.merge(
    conversation_stats,
    on="conversation_id"
)

In [98]:
cases["customer_problem"] = (
    cases["customer_problem"]
    .fillna("")
    .str.strip()
)

cases["amazon_responses"] = (
    cases["amazon_responses"]
    .fillna("")
    .str.strip()
)

cases = cases[
    (cases["customer_problem"] != "") &
    (cases["amazon_responses"] != "") &
    (cases["turn_count"] >= 2)
].copy()

In [99]:
pd.set_option(
    "display.max_colwidth",
    1000
)

cases[
    [
        "conversation_id",
        "turn_count",
        "customer_problem",
        "amazon_responses"
    ]
].sample(
    min(20, len(cases)),
    random_state=42
)

,conversation_id,turn_count,customer_problem,amazon_responses
29043,2133271,8,"@AmazonHelp Ohh yeah, told 4 hours, then told it's a technical error, then told it's 24 hours. 3 different answers. Losing confidence with Amazon.\n@AmazonHelp Pending verification. Nothing has changed in 16 hours. Hence why I am losing confidence.\n@AmazonHelp I ordered 3 sets of Xbox game codes and my account says verifying. They was ordered at 5pm yesterday. Been told 4 hours and still not here.\n@AmazonHelp Been told the wrong thing by 3 different Amazon cx service representative. Why did ibother ordering the digital product from you","@627685 Sorry, that shouldn't take more than 4 hours. Have you received any emails in relation to the order? ^JJ\n@627685 What exactly is the status of the orders now?: https://t.co/aaDyEz1VgE ^PK\n@627685 Please complete the form here: https://t.co/tkLCr7DNil and someone from our team will be in contact about this. ^PK\n@627685 Apologies, can you tell us a bit more w/o sharing any personal or acc info? ^JJ"
24752,180621,12,@AmazonHelp see my account details.u will get every conversation between amazon and me.I have asked some news channel to highlight the same.Amazon=Cheat\n@AmazonHelp always false commitment. Since 18th Sept I am not getting my product.Everyday saying it will be delivered by today.This is really upsetting.\n@AmazonHelp Again false commitment in email.Hi news channels now please take up the matter. Amazon is cheating customers\n@144775 \n@115821 \n@734 https://t.co/djqlLeDiXa\n@AmazonHelp I have already shared my account details. Amazon is only lair and cheater. Please publish my query.\n@144775 \n@734 \n@120320 \n@115821 https://t.co/c4a2MTjDcu\n@AmazonHelp I have replied as you all are lying and cheating.\nAmazon means false commitment and cheating customers.kindly help me by publishing in news.\n@115850 \n@734 \n@12180\n@144775,"@158464 Looks like you had an unpleasant experience with us. Could you please elaborate your query? We'd like to help. ^VM\n@158464 We can't access your account over Twitter. Kindly elaborate your issue, we would like to help. ^CB\n@158464 look into the issue and get back to you. 2/2 ^SH\n@158464 We'll not be able to access your account over twitter. Please share your details and we'll reach out to you. ^HD\n@158464 We've responded to your correspondence to your registered email address.Kindly revert to the email for further assistance.^EM\n@158464 Thanks for confirming that you have reverted to the email. We'll check and get back to you on this accordingly. ^AB"
27705,2057192,9,"@AmazonHelp Why profile of a closed account still floating around on Amazon website?\n@AmazonHelp I closed my acc last year. A few months back I found out that the profile hasn't been removed.\n@AmazonHelp Buyer acc. Contacted numerous times through emails, chats, ph call over 4 months. No1 could help removed it. Just promises.\n@AmazonHelp Sent. Thank you. Hopefully it is resolved this time.\n@AmazonHelp Why are you ignoring my request? 🤔🤔🤔","@608415 I'm sorry for the delayed response. Without posting personal account info, can you tell us what's going on? ^KP\n@608415 Have you recently closed an account with us and still see it active? ^WM\n@608415 Just to confirm, are you referring to a Seller account? When did you last contact us about this? ^EP\n@608415 We'd like to investigate this for you. When you have a moment, please send your info to us here: https://t.co/eQGG2uKPC9 ^JA"
2245,1083868,2,"@AmazonHelp why is it that every time I order something from you guys, I'm bombarded with telesales calls for the next week? Who are you selling my details to?",@375672 I'm sorry to hear this! We'd like to look into this with you! Please reach out to us here: https://t.co/JzP7hlA23B ^ME
77184,823550,12,"@AmazonHelp Sure thing. Here is a screenshot of the promo, from my browser after resetting my password:\n\nThe link to DL did not work. So ... 1/2 https://t.co/Uv3Ze4Fcj3\n@AmazonHelp No, the $5 offer

In [100]:
# Save conversation-level dataset
cases.to_csv(
    "/kaggle/working/amazonhelp_conversations.csv",
    index=False
)

print("Saved: amazonhelp_conversations.csv")
print("Shape:", cases.shape)

Saved: amazonhelp_conversations.csv
Shape: (82556, 8)


In [101]:
# Save tweet-level dataset
amazon_related.to_csv(
    "/kaggle/working/amazonhelp_tweets.csv",
    index=False
)

print("Saved: amazonhelp_tweets.csv")
print("Shape:", amazon_related.shape)

Saved: amazonhelp_tweets.csv
Shape: (353436, 8)
